**# Configuration**

In [0]:
from pyspark.sql.functions import (
    current_timestamp,
    col,
    lit
)

RAW_PATH = "/Volumes/workspace/default/nyc_hvfhv_data/raw/2026-01/"

BRONZE_TABLE = "workspace.default.bronze_hvfhv_trips"

SOURCE_MONTH = "2026-01"

In [0]:
raw_path = "/Volumes/workspace/default/nyc_hvfhv_data/raw/2026-01/"

display(dbutils.fs.ls(raw_path))

In [0]:
df_raw = spark.read.parquet(raw_path)
display(df_raw.limit(10))

In [0]:
df_raw.printSchema()

In [0]:
print(f"Number of columns: {len(df_raw.columns)}")

print("\nColumns:")
for column in df_raw.columns:
    print(column)

In [0]:
display(df_raw.limit(5))

In [0]:
row_count = df_raw.count()
print(f"Total Rows : {row_count}")

In [0]:
from pyspark.sql.functions import min, max

display(
    df_raw.select(
        min("pickup_datetime").alias("min_pickup_datetime"),
        max("pickup_datetime").alias("max-pickup_datetime")
    )
)


In [0]:
from pyspark.sql.functions import col, sum

null_counts = df_raw.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_raw.columns
])

display(null_counts)

**# Added Ingestion Metadata**

In [0]:
df_bronze = (
    df_raw
    .withColumn("_source_month", lit(SOURCE_MONTH))
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingested_at", current_timestamp())
)

In [0]:
(
    df_bronze
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("_source_month")
    .saveAsTable(BRONZE_TABLE)
)

In [0]:
df_bronze_check = spark.read.table(BRONZE_TABLE)

print(f"Bronze Rows: {df_bronze_check.count():,}")
print(f"Bronze Columns: {len(df_bronze_check.columns)}")

display(df_bronze_check.limit(5))

In [0]:
display(
    df_bronze_check.select(
        "hvfhs_license_num",
        "dispatching_base_num",
        "originating_base_num",
        "request_datetime",
        "on_scene_datetime",
        "pickup_datetime",
        "dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_miles",
        "trip_time",
        "base_passenger_fare",
        "tolls",
        "bcf",
        "sales_tax",
        "congestion_surcharge",
        "airport_fee",
        "tips",
        "driver_pay"
    ).limit(20)
)

In [0]:
df_bronze_check.printSchema()

In [0]:
from src.ingestion.bronze import ingest_month_to_bronze

RAW_BASE_PATH = "/Volumes/workspace/default/nyc_hvfhv_data"
BRONZE_TABLE = "workspace.default.bronze_hvfhv_trips"

SOURCE_MONTH = "2026-02"

rows_ingested = ingest_month_to_bronze(
    spark=spark,
    raw_base_path=RAW_BASE_PATH,
    source_month=SOURCE_MONTH,
    bronze_table=BRONZE_TABLE
)

print("Rows ingested:", rows_ingested)

In [0]:
import os

print(os.getcwd())

In [0]:
import sys

REPO_ROOT = "/Workspace/Users/gbpatil2002@gmail.com/nyc-hvfhv-data-platform"

if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

print(REPO_ROOT in sys.path)

In [0]:
from src.ingestion.bronze import ingest_month_to_bronze

print("Bronze module imported successfully!")

In [0]:
import os

REPO_ROOT = "/Workspace/Users/gbpatil2002@gmail.com/nyc-hvfhv-data-platform"

print(os.path.exists(REPO_ROOT))
print(os.path.exists(f"{REPO_ROOT}/src"))
print(os.path.exists(f"{REPO_ROOT}/src/ingestion"))
print(os.path.exists(f"{REPO_ROOT}/src/ingestion/bronze.py"))

In [0]:
dbutils.library.restartPython()

In [0]:
import sys

SRC_PATH = "/Workspace/Users/gbpatil2002@gmail.com/nyc-hvfhv-data-platform/src"

if SRC_PATH not in sys.path:
    sys.path.append(SRC_PATH)

print(SRC_PATH in sys.path)

In [0]:
from ingestion.bronze import ingest_month_to_bronze

print("Bronze module imported successfully!")

In [0]:
RAW_BASE_PATH = "/Volumes/workspace/default/nyc_hvfhv_data"

BRONZE_TABLE = "workspace.default.bronze_hvfhv_trips"

SOURCE_MONTH = "2026-02"

rows_ingested = ingest_month_to_bronze(
    spark=spark,
    raw_base_path=RAW_BASE_PATH,
    source_month=SOURCE_MONTH,
    bronze_table=BRONZE_TABLE
)

print("Rows ingested:", rows_ingested)

In [0]:
from pyspark.sql.functions import col

df_bronze = spark.table(
    "workspace.default.bronze_hvfhv_trips"
)

print(
    "January:",
    df_bronze
    .filter(col("_source_month") == "2026-01")
    .count()
)

print(
    "February:",
    df_bronze
    .filter(col("_source_month") == "2026-02")
    .count()
)

print(
    "Total:",
    df_bronze.count()
)